In [1]:
import pandas as pd
import sqlite3

In [2]:
# conectamos con la base de datos my_database.db
connection = sqlite3.connect("database_cervezas.db")

# obtenemos un cursor que utilizamos para las queries
crsr = connection.cursor()

In [3]:
# Con esta función leemos los datos y lo pasamos a un DataFrame de Pandas
def sql_query(query):

    # Ejecuta la query
    crsr.execute(query)

    # Almacena los datos de la query 
    ans = crsr.fetchall()

    # Obtenemos los nombres de las columnas de la tabla
    names = [description[0] for description in crsr.description]

    return pd.DataFrame(ans,columns=names)

In [4]:
res = crsr.execute("SELECT name FROM sqlite_master WHERE type='table'")
for name in res:
    print(name[0])

CERVEZAS


In [5]:
query = '''
CREATE TABLE CERVEZAS (
    CodC INT PRIMARY KEY,
    Envase VARCHAR,
    Capacidad FLOAT,
    Stock INT
);
'''

crsr.execute(query)

OperationalError: table CERVEZAS already exists

In [6]:
query = '''
INSERT INTO CERVEZAS (CodC, Envase, Capacidad, Stock)
VALUES
('01', 'Botella', 0.20, 3600),
('02', 'Botella', 0.33, 1200),
('03', 'Lata', 0.33, 2400),
('04', 'Botella', 1.00, 288),
('05', 'Barril', 60.00, 30);
'''

crsr.execute(query)

In [8]:
query = '''
CREATE TABLE BARES (
    CodB INT PRIMARY KEY,
    Nombre VARCHAR,
    Cif VARCHAR,
    Localidad VARCHAR
);

'''

crsr.execute(query)

In [9]:
query = '''
INSERT INTO BARES (CodB, Nombre, Cif, Localidad)
VALUES
('001', 'Stop', '11111111X', 'Villa Botijo'),
('002', 'Las Vegas', '22222222Y', 'Villa Botijo'),
('003', 'Club Social', NULL, 'Las Ranas'),
('004', 'Otra Ronda', '33333333Z', 'La Esponja');
'''

crsr.execute(query)

In [11]:
query = '''
CREATE TABLE EMPLEADOS (
    CodE INT PRIMARY KEY,
    Nombre VARCHAR,
    Sueldo INT
);
'''
crsr.execute(query)

In [12]:
query = '''
INSERT INTO EMPLEADOS (CodE, Nombre, Sueldo) VALUES
(1, 'Carlos Lopez', 120000),
(2, 'Rosa Perez', 110000),
(3, 'Luisa Garcia', 100000);
'''
crsr.execute(query)

In [14]:
query = '''
CREATE TABLE REPARTO (
    CodE INT,
    CodB INT,
    CodC INT,
    Fecha DATE,
    Cantidad INT,
    PRIMARY KEY (CodE, CodB, CodC, Fecha),
    FOREIGN KEY (CodE) REFERENCES EMPLEADOS(CodE),
    FOREIGN KEY (CodB) REFERENCES BARES(CodB),
    FOREIGN KEY (CodC) REFERENCES CERVEZAS(CodC)
);
'''
# CRUD Create Register(insert) Update Delete
crsr.execute(query)

In [15]:
query = '''
INSERT INTO REPARTO (CodE, CodB, CodC, Fecha, Cantidad) VALUES
(1, '001', '01', '2005-10-21', 240),
(1, '001', '02', '2005-10-21', 48),
(1, '002', '03', '2005-10-22', 60),
(1, '004', '05', '2005-10-22', 4),
(2, '002', '03', '2005-10-22', 48),
(2, '002', '05', '2005-10-23', 2),
(2, '004', '01', '2005-10-23', 480),
(2, '004', '02', '2005-10-24', 72),
(3, '003', '03', '2005-10-24', 48),
(3, '003', '04', '2005-10-25', 20);
'''
crsr.execute(query)

In [ ]:
query = '''

'''
crsr.execute(query)

In [ ]:
query = '''

'''
crsr.execute(query)

In [101]:
query = '''
SELECT * FROM CERVEZAS
'''
sql_query(query)

,CodC,Envase,Capacidad,Stock
0,1,Botella,0.20,3600
1,2,Botella,0.33,1200
2,3,Lata,0.33,2400
3,4,Botella,1.00,288
4,5,Barril,60.00,30


In [22]:
# 1 Obtener el nombre de los empleados que hayan repartido al bar Stop durante la semana 
# del 17 al 23 de octubre de 2005.

query = '''
SELECT DISTINCT e.Nombre
FROM REPARTO r
JOIN EMPLEADOS e ON r.CodE = e.CodE
JOIN BARES b ON r.CodB = b.CodB
WHERE b.Nombre = 'Stop'
  AND r.Fecha BETWEEN '2005-10-17' AND '2005-10-23';
'''

sql_query(query)

,Nombre
0,Carlos Lopez


In [37]:
# 2 Obtener el Cif y nombre de los bares a los que se ha repartido cerveza de tipo Botella y
# capacidad inferior a 1 litro, ordenados por localidad

query = '''
SELECT DISTINCT b.Cif, b.Nombre
FROM REPARTO r
JOIN CERVEZAS c ON r.CodC = c.CodC
JOIN BARES b ON r.CodB = b.CodB
WHERE c.Envase = "Botella"
AND c.Capacidad < 1
ORDER BY b.Localidad
'''

sql_query(query)

,Cif,Nombre
0,33333333Z,Otra Ronda
1,11111111X,Stop


In [ ]:
# 3. Obtener los repartos (nombre del bar, envase y capacidad de la bebida, fecha y cantidad)

query = '''
SELECT
    b.Nombre AS Bar,
    c.Envase, c.Capacidad,
    r.Fecha,
    r.Cantidad
FROM REPARTO r
JOIN CERVEZAS c ON r.CodC = c.CodC
JOIN BARES b ON r.CodB = b.CodB
'''

sql_query(query)

,Bar,Envase,Capacidad,Fecha,Cantidad
0,Stop,Botella,0.20,2005-10-21,240
1,Stop,Botella,0.33,2005-10-21,48
2,Las Vegas,Lata,0.33,2005-10-22,60
3,Otra Ronda,Barril,60.00,2005-10-22,4
4,Las Vegas,Lata,0.33,2005-10-22,48
5,Las Vegas,Barril,60.00,2005-10-23,2
6,Otra Ronda,Botella,0.20,2005-10-23,480
7,Otra Ronda,Botella,0.33,2005-10-24,72
8,Club Social,Lata,0.33,2005-10-24,48
9,Club Social,Botella,1.00,2005-10-25,20


In [51]:
# 4. Obtener los bares a los que se les ha repartido envases de tipo botella y capacidad 0.2 o
# 0.33

query = '''
SELECT DISTINCT
    b.Nombre AS Bar
FROM REPARTO r
JOIN CERVEZAS c ON r.CodC = c.CodC
JOIN BARES b ON r.CodB = b.CodB
WHERE 
    c.Envase = "Botella"
    AND (c.Capacidad = 0.2 OR c.Capacidad = 0.33)
'''
sql_query(query)

,Bar
0,Stop
1,Otra Ronda


In [63]:
# 5. Nombre de los empleados que han repartido a los bares "Stop" y "Las Vegas" cervezas con
# envase botella.

query = '''
SELECT DISTINCT
    e.Nombre AS Empleado
FROM REPARTO r
JOIN CERVEZAS c ON r.CodC = c.CodC
JOIN BARES b ON r.CodB = b.CodB
JOIN EMPLEADOS e ON r.CodE = e.CodE
WHERE
    (b.Nombre = "Stop" OR b.Nombre = "Las Vegas")
    AND c.Envase = "Botella"
'''
sql_query(query)

,Empleado
0,Carlos Lopez


In [90]:
# 6. Obtener el nombre y número de viajes que ha realizado cada empleado fuera de Villa
# Botijo.

query = '''
SELECT 
    e.Nombre AS Empleado,
    COUNT (*) AS NumViajes
FROM REPARTO r
JOIN BARES b ON r.CodB = b.CodB
JOIN EMPLEADOS e ON r.CodE = e.CodE
WHERE b.Localidad <> "Villa Botijo"
GROUP BY e.Nombre
'''
sql_query(query)

,Empleado,NumViajes
0,Carlos Lopez,1
1,Luisa Garcia,2
2,Rosa Perez,2


In [115]:
# 7. Obtener el nombre y localidad del bar que más litros de cerveza ha comprado.

query = '''
SELECT
    SUM (r.Cantidad * c.Capacidad) AS Litros,
    b.Nombre AS Bar,
    b.Localidad
FROM REPARTO r
JOIN BARES b ON r.CodB = b.CodB
JOIN CERVEZAS c ON r.CodC = c.CodC
GROUP BY b.Nombre, b.Localidad
ORDER BY Litros DESC
LIMIT 1
'''
sql_query(query)

,Litros,Bar,Localidad
0,359.76,Otra Ronda,La Esponja


In [134]:
# 8. Obtener los bares que han adquirido todos los tipos de cerveza con envase de botella y
# capacidad menor que 1 litro.

query = '''
SELECT
    b.Nombre AS Bar
FROM REPARTO r
JOIN BARES b ON r.CodB = b.CodB
JOIN CERVEZAS c ON r.CodC = c.CodC
WHERE
    c.Envase = "Botella"
    AND c.Capacidad < 1
GROUP BY b.Nombre
HAVING COUNT(DISTINCT c.CodC) = (
    SELECT COUNT(*) 
    FROM CERVEZAS 
    WHERE Envase = "Botella" AND Capacidad < 1
);
'''
sql_query(query)

,Bar
0,Otra Ronda
1,Stop


In [163]:
#9. Subir un 5% el sueldo del empleado que más días haya trabajado.

query = '''
UPDATE EMPLEADOS
SET Sueldo = Sueldo * 1.05
WHERE CodE = (
    SELECT CodE
    FROM REPARTO
    GROUP BY CodE
    ORDER BY COUNT(*) DESC
    LIMIT 1
);
'''
sql_query(query)

TypeError: 'NoneType' object is not iterable

In [164]:
query = '''
SELECT COUNT (*) AS Pedidos, e.codE, e.Nombre, e.Sueldo
FROM REPARTO r
JOIN EMPLEADOS e ON r.CodE = e.CodE
GROUP BY e.Nombre
ORDER BY Pedidos DESC
LIMIT 1
'''
sql_query(query)

,Pedidos,CodE,Nombre,Sueldo
0,4,2,Rosa Perez,115500


In [ ]:
query = '''
   
'''
sql_query(query)

In [ ]:
query = '''

'''
crsr.execute(query)

In [ ]:
query = '''

'''
sql_query(query)